In [17]:
# ================= INSTALL =================
!pip install timm -q
!pip install grad-cam -q

# ================= IMPORTS =================
import os, cv2, numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix, 
                             classification_report, roc_curve, auc)
import matplotlib.pyplot as plt
import seaborn as sns
import timm
import warnings
warnings.filterwarnings("ignore")

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# ================= CONFIG =================
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 20
LR = 5e-5 # Lower LR is better for Transformers
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# ================= AUTO FIND DATA =================
def find_dir(name, start='/kaggle/input'):
    for root, dirs, files in os.walk(start):
        if name in dirs:
            return os.path.join(root, name)
    return None

train_img_dir = find_dir('X_train')
train_mask_dir = find_dir('Y_train_Cancer')
val_img_dir   = find_dir('X_test')
val_mask_dir   = find_dir('Y_test_Cancer')

train_img = sorted([os.path.join(train_img_dir, f) for f in os.listdir(train_img_dir)])
train_mask = sorted([os.path.join(train_mask_dir, f) for f in os.listdir(train_mask_dir)])
val_img = sorted([os.path.join(val_img_dir, f) for f in os.listdir(val_img_dir)])
val_mask = sorted([os.path.join(val_mask_dir, f) for f in os.listdir(val_mask_dir)])

# ================= DATASET =================
class PancreasDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx], cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
        
        # Enhanced Contrast for Pancreas
        img = self.clahe.apply(img)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)).astype(np.float32)/255.0
        img = np.stack([img]*3, axis=0) # 3-channel for Swin

        label = 1 if np.any(mask > 0) else 0
        img = torch.tensor(img, dtype=torch.float32)
        
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long)

# ================= AUGMENTATION & LOADERS =================
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds = PancreasDataset(train_img, train_mask, train_tf)
val_ds = PancreasDataset(val_img, val_mask, val_tf)

# Correct Balancing Logic
train_labels = [1 if np.any(cv2.imread(m,0)>0) else 0 for m in train_mask]
counts = np.bincount(train_labels)
weights = 1. / torch.tensor(counts, dtype=torch.float)
sample_weights = weights[train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# ================= SWIN TRANSFORMER MODEL =================
class SwinPancreas(nn.Module):
    def __init__(self):
        super().__init__()
        # Using Swin-Tiny (Faster and often more accurate on small datasets)
        self.model = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=2)

    def forward(self, x):
        return self.model(x)

model = SwinPancreas().to(DEVICE)

# ================= TRAINING SETUP =================
# Label smoothing helps prevent overconfidence on blurry CT boundaries
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# ================= TRAINING LOOP =================
best_f1 = 0
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    preds, gts = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            preds.extend(outputs.argmax(1).cpu().numpy())
            gts.extend(labels.cpu().numpy())

    f1 = f1_score(gts, preds)
    acc = accuracy_score(gts, preds)
    print(f"Epoch {epoch+1} | Loss: {train_loss/len(train_loader):.4f} | Acc: {acc:.4f} | F1: {f1:.4f}")
    
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "best_swin_pancreas.pth")
    scheduler.step()

# ================= EVALUATION =================
print("\nFinal Classification Report:")
print(classification_report(gts, preds, target_names=['Healthy', 'Tumor']))

# ================= GRAD-CAM =================
# Swin Grad-CAM looks at the last Norm layer of the last block
target_layers = [model.model.norm]
cam = GradCAM(model=model, target_layers=target_layers)
img, label = val_ds[0]
input_tensor = img.unsqueeze(0).to(DEVICE)
grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(1)])[0]
visualization = show_cam_on_image(img.numpy().transpose(1,2,0), grayscale_cam, use_rgb=True)

plt.imshow(visualization)
plt.title("Swin Attention Heatmap (Tumor Focus)")
plt.show()

Using device: cuda


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Epoch 1 | Loss: 0.5424 | Acc: 0.8376 | F1: 0.8874
Epoch 2 | Loss: 0.4858 | Acc: 0.8207 | F1: 0.8742
Epoch 3 | Loss: 0.4544 | Acc: 0.7907 | F1: 0.8481
Epoch 4 | Loss: 0.4195 | Acc: 0.8104 | F1: 0.8647
Epoch 5 | Loss: 0.4016 | Acc: 0.7886 | F1: 0.8444
Epoch 6 | Loss: 0.3868 | Acc: 0.8104 | F1: 0.8656
Epoch 7 | Loss: 0.3738 | Acc: 0.7602 | F1: 0.8198
Epoch 8 | Loss: 0.3577 | Acc: 0.8289 | F1: 0.8832
Epoch 9 | Loss: 0.3552 | Acc: 0.8087 | F1: 0.8638
Epoch 10 | Loss: 0.3369 | Acc: 0.8229 | F1: 0.8774
Epoch 11 | Loss: 0.3344 | Acc: 0.8196 | F1: 0.8747
Epoch 12 | Loss: 0.3279 | Acc: 0.8272 | F1: 0.8810
Epoch 13 | Loss: 0.3156 | Acc: 0.8213 | F1: 0.8745
Epoch 14 | Loss: 0.2981 | Acc: 0.8234 | F1: 0.8771
Epoch 15 | Loss: 0.3015 | Acc: 0.8283 | F1: 0.8822
Epoch 16 | Loss: 0.2900 | Acc: 0.8267 | F1: 0.8810
Epoch 17 | Loss: 0.2880 | Acc: 0.8305 | F1: 0.8827
Epoch 18 | Loss: 0.2859 | Acc: 0.8311 | F1: 0.8833
Epoch 19 | Loss: 0.2828 | Acc: 0.8294 | F1: 0.8825
Epoch 20 | Loss: 0.2853 | Acc: 0.8300 | 

Exception: The input image should np.float32 in the range [0, 1]